# 01 — Data Loading & Preprocessing
**DSA210 Term Project · Nehir Eylül Balcı**

Loads the four data sources, standardises country names, merges into a single dataset, and applies log transformations.

In [10]:
from google.colab import files
from IPython.display import display
import re, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## Files to upload
- `FIVB_Volleyball_Rankings.xlsx`
- `API_NY.GDP.PCAP.CD_DS2...zip` — GDP *per capita* (not MKTP)
- `API_SP.POP.TOTL_DS2...zip` — Population
- `HDR25_Statistical_Annex_HDI_Table.xlsx` — HDI

In [11]:
uploaded = files.upload()

ranking_file = gdp_zip = pop_zip = hdi_file = None
for fn in uploaded:
    fl = fn.lower()
    if "fivb" in fl and fn.endswith(".xlsx"):                  ranking_file = fn
    elif "pcap" in fl and fn.endswith(".zip"):                 gdp_zip = fn
    elif "sp.pop" in fl and fn.endswith(".zip"):               pop_zip = fn
    elif ("hdi" in fl or "hdr" in fl) and fn.endswith(".xlsx"): hdi_file = fn

missing = [n for n,f in [("FIVB",ranking_file),("GDP",gdp_zip),
                          ("Pop",pop_zip),("HDI",hdi_file)] if not f]
if missing:
    raise ValueError(f"Missing: {missing}")
print("all files found ✓")

Saving API_NY.GDP.PCAP.CD_DS2_en_csv_v2_245.zip to API_NY.GDP.PCAP.CD_DS2_en_csv_v2_245 (1).zip
Saving API_SP.POP.TOTL_DS2_en_csv_v2_58.zip to API_SP.POP.TOTL_DS2_en_csv_v2_58 (1).zip
Saving FIVB_Volleyball_Rankings.xlsx to FIVB_Volleyball_Rankings (1).xlsx
Saving HDR25_Statistical_Annex_HDI_Table.xlsx to HDR25_Statistical_Annex_HDI_Table (1).xlsx
all files found ✓


## 1.1 FIVB Rankings

In [12]:
NAME_FIX = {
    "Turkey":"Turkiye", "South Korea":"Korea, Rep.",
    "Iran":"Iran, Islamic Rep.", "Czech Republic":"Czechia",
    "Slovakia":"Slovak Republic", "Vietnam":"Viet Nam",
    "Egypt":"Egypt, Arab Rep.",
}

def load_fivb(path, sheet):
    raw = pd.read_excel(path, sheet_name=sheet, header=1)
    col = {}
    for c in raw.columns:
        lc = str(c).strip().lower()
        if lc == "rank":          col[c] = "rank"
        elif lc == "country":     col[c] = "country"
        elif "point" in lc:       col[c] = "points"
        elif "confed" in lc:      col[c] = "confederation"
    raw = raw.rename(columns=col)
    df = raw[[c for c in ["rank","country","points","confederation"] if c in raw.columns]].copy()
    df["rank"]   = pd.to_numeric(df["rank"],   errors="coerce")
    df["points"] = pd.to_numeric(df["points"], errors="coerce")
    df = df.dropna(subset=["rank","country"])
    df["country_std"] = df["country"].astype(str).str.strip().replace(NAME_FIX)
    return df

men   = load_fivb(ranking_file, "Senior Men")
women = load_fivb(ranking_file, "Senior Women")

vb = (men[["country_std","points","confederation"]]
      .rename(columns={"points":"pts_men"})
      .merge(women[["country_std","points"]].rename(columns={"points":"pts_women"}),
             on="country_std", how="outer"))

vb["avg_senior_pts"] = vb[["pts_men","pts_women"]].mean(axis=1)
vb["gender_gap"]     = vb["pts_women"] - vb["pts_men"]

print(f"{len(vb)} countries | men: {vb['pts_men'].notna().sum()} | women: {vb['pts_women'].notna().sum()}")
display(vb.head(8))

38 countries | men: 30 | women: 30


,country_std,pts_men,confederation,pts_women,avg_senior_pts,gender_gap
0,Argentina,266.94,CSV,180.96,223.950,-85.98
1,Belgium,183.24,CEV,190.98,187.110,7.74
2,Brazil,305.87,CSV,417.92,361.895,112.05
3,Bulgaria,161.06,CEV,169.78,165.420,8.72
4,Canada,254.46,NORCECA,245.38,249.920,-9.08
5,Chile,139.14,CSV,NaN,139.140,NaN
6,China,144.02,AVC,346.75,245.385,202.73
7,Colombia,NaN,NaN,152.37,152.370,NaN


## 1.2 World Bank — GDP & Population

In [13]:
def read_wb_zip(zf):
    with zipfile.ZipFile(zf) as z:
        csv = [f for f in z.namelist() if f.endswith(".csv") and "Metadata" not in f][0]
        df  = pd.read_csv(z.open(csv), skiprows=4)
    # find year columns by checking if column name is a 4-digit number
    yrs = [c for c in df.columns if str(c).strip().isdigit() and len(str(c).strip()) == 4]
    df[yrs] = df[yrs].apply(pd.to_numeric, errors="coerce")
    df = df[df["Country Code"].astype(str).str.len() == 3].copy()
    df["value"] = df[yrs].ffill(axis=1).iloc[:, -1]
    return df.rename(columns={"Country Name":"country_std"})[["country_std","value"]]

gdp = read_wb_zip(gdp_zip).rename(columns={"value":"gdp_per_capita"})
pop = read_wb_zip(pop_zip).rename(columns={"value":"population"})
print(f"GDP: {len(gdp)} rows | Pop: {len(pop)} rows")

GDP: 266 rows | Pop: 266 rows


## 1.3 UNDP — HDI

In [14]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    try:    hdi_raw = pd.read_excel(hdi_file, sheet_name="Table 1. HDI", header=5)
    except: hdi_raw = pd.read_excel(hdi_file, header=5)

c_col = next((c for c in hdi_raw.columns if str(c).strip().lower()=="country"), None)
v_col = next((c for c in hdi_raw.columns if str(c).strip().lower() in ["value","hdi","hdi value"]), None)
if v_col is None:
    for c in hdi_raw.columns:
        if c == c_col: continue
        s = pd.to_numeric(hdi_raw[c], errors="coerce").dropna()
        if len(s) > 10 and s.between(0,1).mean() > 0.5:
            v_col = c; break

hdi = hdi_raw[[c_col, v_col]].copy()
hdi.columns = ["country","hdi"]
hdi["hdi"] = pd.to_numeric(hdi["hdi"], errors="coerce")
hdi = hdi.dropna(subset=["country","hdi"])
HDI_FIX = {"Türkiye":"Turkiye","Iran (Islamic Republic of)":"Iran, Islamic Rep.",
            "Korea (Republic of)":"Korea, Rep.","Slovakia":"Slovak Republic",
            "Viet Nam":"Viet Nam","Egypt":"Egypt, Arab Rep."}
hdi["country_std"] = hdi["country"].astype(str).str.strip().replace(HDI_FIX)
hdi = hdi[["country_std","hdi"]]
print(f"HDI: {len(hdi)} rows")

HDI: 208 rows


## 1.4 Merge & Feature Engineering

In [15]:
df = (vb
      .merge(gdp, on="country_std", how="left")
      .merge(pop, on="country_std", how="left")
      .merge(hdi, on="country_std", how="left")
      .drop_duplicates("country_std")
      .reset_index(drop=True))

# log transform — GDP and population are right-skewed
df["log_gdp"]        = np.log10(df["gdp_per_capita"])
df["log_population"] = np.log10(df["population"])

print(f"final dataset: {df.shape[0]} countries × {df.shape[1]} columns")
print(f"NaN summary:\n{df[['pts_men', 'pts_women', 'log_gdp', 'log_population', 'hdi']].isna().sum()}")
display(df.describe().round(2))

df.to_csv("volleyball_economic_dataset.csv", index=False)
print("saved ✓")

final dataset: 38 countries × 11 columns
NaN summary:
pts_men           8
pts_women         8
log_gdp           1
log_population    1
hdi               1
dtype: int64


,pts_men,pts_women,avg_senior_pts,gender_gap,gdp_per_capita,population,hdi,log_gdp,log_population
count,30.00,30.00,38.00,22.00,37.00,3.700000e+01,37.00,37.00,37.00
mean,221.03,225.68,207.49,7.92,28307.59,9.138988e+07,0.86,4.30,7.48
std,87.39,94.99,77.34,104.77,22024.07,2.325201e+08,0.08,0.40,0.62
min,125.41,127.51,127.51,-209.81,2132.43,2.127400e+06,0.63,3.33,6.33
25%,146.83,152.21,148.24,-31.14,10875.66,1.069468e+07,0.80,4.04,7.03
50%,184.16,185.85,186.09,0.50,23292.33,3.786022e+07,0.88,4.37,7.58
75%,272.52,260.45,247.38,55.17,40385.34,8.351659e+07,0.92,4.61,7.92
max,401.31,456.91,401.57,202.73,84534.04,1.408975e+09,0.96,4.93,9.15


saved ✓


In [16]:
from google.colab import files
files.download('volleyball_economic_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>